# Text Classification — Few-Shot with Groq

In [1]:
import os
import time
import random
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from groq import Groq
from difflib import get_close_matches

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

env_candidates = [
    Path(".env"),
    Path("shot") / ".env",
]
loaded = False
for env_path in env_candidates:
    if env_path.exists():
        if load_dotenv is not None:
            load_dotenv(env_path)
        else:
            for line in env_path.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                k, v = line.split("=", 1)
                os.environ[k.strip()] = v.strip().strip('"').strip("'")
        print(f"Loaded .env from: {env_path.resolve()}")
        loaded = True
        break
if not loaded:
    print("No .env file found in expected paths (.env, shot/.env).")

# -- CONFIG -----------------------------------------------------------------
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
MODEL = "llama-3.3-70b-versatile"
K_SHOTS = 3
SLEEP_SEC = 1.2
RANDOM_SEED = 808815
# ---------------------------------------------------------------------------

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found. Add it to .env in this folder (or shot/.env).")

random.seed(RANDOM_SEED)
client = Groq(api_key=GROQ_API_KEY)
print("Groq client ready")

Loaded .env from: C:\Users\bruno\Desktop\um_ucs_mestrado\deep_learning\projeto\shot\.env
Groq client ready


## 1. Load Data


In [2]:
# -- Labeled training data ---------------------------------------------------
# Expected columns: ID, Text, Label (adjust if column names differ)

TRAIN_FILES = {
    "train1": "subm1_labels_revealed.csv",
    "train2": "subm2_labels_revealed.csv",
}

train_dfs = {}
for name, path in TRAIN_FILES.items():
    train_df_ = pd.read_csv(path, sep=";", encoding="utf-8-sig")
    train_dfs[name] = train_df_
    print(f"{name}: {len(train_df_)} rows")

train_df = pd.concat(train_dfs.values(), ignore_index=True)
print(f"\nTotal training rows: {len(train_df)}")

train1: 100 rows
train2: 100 rows

Total training rows: 200


In [ ]:

FILES = {

    "subm3": "subm3.csv",
}

dfs = {}
for name, path in FILES.items():
    df = pd.read_csv(path, sep=";", encoding="utf-8-sig")
    dfs[name] = df
    print(f"{name}: {len(df)} rows — columns: {list(df.columns)}")

# Combine all into one DataFrame for batch processing
all_df = pd.concat(dfs.values(), ignore_index=True)
print(f"\nTotal rows: {len(all_df)}")
all_df.head()

subm3: 150 rows — columns: ['ID', 'Text']

Total rows: 150


,ID,Text
0,D2-126,The reality about the places that diamonds are...
1,D2-127,Geothermobarometric calculations for a worldwi...
2,D2-128,Diamonds are formed deep within the Earth’s ma...
3,D2-129,Diamond is a solid form of the element carbon ...
4,D2-130,Diamonds are formed deep within the Earth unde...


## 2. Build Examples Pool


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train_df["Text"] = train_df["Text"].fillna("").astype(str)
train_df["Label"] = train_df["Label"].astype(str).str.strip()
LABELS = sorted(train_df["Label"].dropna().unique().tolist())
VALID_LABELS_LOWER = {l.lower(): l for l in LABELS}
print(f"All labels ({len(LABELS)}): {LABELS[:10]} ...")

print("Building TF-IDF index over training set...")
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
train_tfidf = vectorizer.fit_transform(train_df["Text"])
print(f"TF-IDF matrix: {train_tfidf.shape}")

All labels (5): ['Anthropic', 'Google', 'Human', 'Meta', 'OpenAI'] ...
Building TF-IDF index over training set...
TF-IDF matrix: (200, 10000)


In [7]:
def get_similar_examples(query_text: str, k: int = K_SHOTS) -> list[dict]:
    """
    Retrieve k training examples that are most similar to the query text.
    Returns a list of dicts with 'text' and 'label' keys.
    """
    query_text = "" if query_text is None else str(query_text)
    query_vec = vectorizer.transform([query_text])
    sims = cosine_similarity(query_vec, train_tfidf).flatten()
    top_k_idx = sims.argsort()[::-1][:k]

    examples = []
    for idx in top_k_idx:
        row = train_df.iloc[idx]
        examples.append({"text": row["Text"], "label": row["Label"]})
    return examples


# Fallback: random examples when TF-IDF is not available or for diversity
def get_random_examples(k: int = K_SHOTS) -> list[dict]:
    """Sample k random training examples, one per label for diversity."""
    sampled = train_df.groupby("Label").apply(
        lambda g: g.sample(1, random_state=RANDOM_SEED)
    ).sample(min(k, len(LABELS)), random_state=RANDOM_SEED)
    return [{"text": r["Text"], "label": r["Label"]} for _, r in sampled.iterrows()]


# Test retrieval
test_query = all_df["Text"].iloc[0]
examples = get_similar_examples(test_query)
print("Examples retrieved for first query:")
for i, ex in enumerate(examples, 1):
    print(f"  {i}. [{ex['label']}] {ex['text'][:100]}...")

Examples retrieved for first query:
  1. [Meta] Diamonds are formed through a natural geological process that involves high pressure and temperature...
  2. [Google] Diamonds, those captivating gemstones, are forged under truly extreme conditions deep within the Ear...
  3. [Human] High-pressure areas form due to downward motion through the troposphere, the atmospheric layer where...


## 3. Few-Shot Classification Function

In [8]:
LABELS_STR = "\n".join(f"- {l}" for l in LABELS)

SYSTEM_PROMPT = """You are a precise text classification assistant.
Classify each text into exactly one label from the provided label list.
Use only labels from the list and output only the label name (no extra text)."""


def format_examples(examples: list[dict]) -> str:
    """Format retrieved examples for inclusion in the prompt."""
    parts = []
    for i, ex in enumerate(examples, 1):
        parts.append(
            f"Example {i}:\n"
            f"Text: \"{str(ex['text'])[:500]}\"\n"
            f"Label: {str(ex['label']).strip()}"
        )
    return "\n\n".join(parts)


def normalize_pred_label(pred: str) -> str:
    pred_clean = str(pred).strip().strip('"\'`').rstrip(".,;:").strip()
    return VALID_LABELS_LOWER.get(pred_clean.lower(), pred_clean)


def classify_few_shot(text: str, use_similarity: bool = True) -> str:
    """Classify a single text using few-shot prompting."""
    text = "" if text is None else str(text)

    if use_similarity:
        examples = get_similar_examples(text, k=K_SHOTS)
    else:
        examples = get_random_examples(k=K_SHOTS)

    examples_str = format_examples(examples)

    user_prompt = f"""Valid labels:

{LABELS_STR}

Labeled examples:

{examples_str}

Text to classify:
\"\"\"
{text[:3000]}
\"\"\"

Return exactly one label from the valid labels list."""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        max_tokens=20,
    )
    raw = response.choices[0].message.content
    return normalize_pred_label(raw)


# Quick test
print("Test prediction:", classify_few_shot(all_df["Text"].iloc[0]))

Test prediction: OpenAI


## 4. Run Inference on All Rows

In [8]:
predictions = []
errors      = []

for idx, row in tqdm(all_df.iterrows(), total=len(all_df), desc="Classifying (few-shot)"):
    try:
        label = classify_few_shot(row["Text"], use_similarity=True)
        predictions.append({"ID": row["ID"], "Label": label})
    except Exception as e:
        print(f"Error on ID {row['ID']}: {e}")
        errors.append(row["ID"])
        predictions.append({"ID": row["ID"], "Label": "ERROR"})
    time.sleep(SLEEP_SEC)

print(f"\nDone. Errors: {len(errors)}")

Classifying (few-shot): 100%|██████████| 150/150 [07:49<00:00,  3.13s/it]


Done. Errors: 0


## 5. Post-process: Snap to Closest Valid Label

In [20]:
results_df = pd.DataFrame(predictions)

def snap_label(pred: str, valid_labels: list, cutoff: float = 0.6) -> str:
    pred = normalize_pred_label(pred)
    if pred in valid_labels:
        return pred
    matches = get_close_matches(pred, valid_labels, n=1, cutoff=cutoff)
    return matches[0] if matches else pred

results_df["Label_snapped"] = results_df["Label"].apply(
    lambda x: snap_label(x, LABELS)
)

corrections = results_df[results_df["Label"] != results_df["Label_snapped"]]
print(f"Corrections made: {len(corrections)}")
corrections[["ID", "Label", "Label_snapped"]].head(10)

NameError: name 'predictions' is not defined

## 6. Save Submissions

In [ ]:
final_df = results_df[["ID", "Label_snapped"]].rename(columns={"Label_snapped": "Label"})

out_path = "sub3-g5-MECD-A.csv"
final_df.to_csv(out_path, sep=";", index=False)
print(f"Saved {len(final_df)} rows -> {out_path}")

final_df.head(10)

Saved 150 rows -> submission_fewshot_all.csv


,ID,Label
0,D2-126,OpenAI
1,D2-127,Meta
2,D2-128,OpenAI
3,D2-129,OpenAI
4,D2-130,Meta
5,D2-131,Meta
6,D2-132,Human
7,D2-133,Meta
8,D2-134,Meta
9,D2-135,Meta
